In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import subprocess
import sys
from urllib.request import urlretrieve, urlopen
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path("..").resolve()))

from _simulation_workers import tune_r
from geomexp.visualization import ClusterVisualizer, PlotStyle

warnings.filterwarnings("ignore")

for pkg in ["pandas", "xarray", "h5netcdf", "h5py"]:
    try:
        __import__(pkg)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

import pandas as pd
import xarray as xr

In [ ]:
_viridis = mpl.colormaps["viridis"]
_viridis_colors = [mpl.colors.to_hex(_viridis(i / 4)) for i in range(5)]

style = PlotStyle(figsize=(6.5, 3.5), use_latex=True, fontsize=12, color_palette=_viridis_colors)
try:
    viz = ClusterVisualizer(style)
except RuntimeError:
    style = PlotStyle(figsize=(6.5, 3.5), use_latex=False, fontsize=12, color_palette=_viridis_colors)
    viz = ClusterVisualizer(style)

def apply_plotstyle() -> ClusterVisualizer:
    return ClusterVisualizer(style)

viz = apply_plotstyle()

PLOT_DIR = Path("../plots/case_study")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = Path("../results/case_study")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("../results/case_study/goes_raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
K = 3
N_RESTARTS = 20
M_GRID = 120
APPLY_INTERPOLATION = True
APPLY_L2_NORMALIZATION = True
N_YEARS_TARGET = 10
TARGET_SATELLITES = ["GOES-18", "GOES-19"]
MAX_EVENTS_PER_SATELLITE = 10000000
MIN_DURATION_MIN = 10
MAX_DURATION_MIN = 240

GRID = np.linspace(0, 1, M_GRID)
RNG = np.random.default_rng(SEED)

In [ ]:
def _fetch_text(url: str) -> str:
    with urlopen(url) as response:
        return response.read().decode("utf-8")


def available_yearly_urls(sat: str) -> dict[int, str]:
    base = (
        f"https://data.ngdc.noaa.gov/platforms/solar-space-observing-satellites/goes/"
        f"goes{sat}/l2/data/xrsf-l2-avg1m_science/"
    )
    html = _fetch_text(base)
    pat = re.compile(r'href="(sci_xrsf-l2-avg1m_g' + sat + r'_y(\d{4})_v[\d\-]+\.nc)"')
    out: dict[int, str] = {}
    for fname, year in pat.findall(html):
        out[int(year)] = base + fname
    return dict(sorted(out.items()))


def download(url: str, dst: Path) -> Path:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        urlretrieve(url, dst)
    return dst


def load_xrsb_flux(paths: list[Path]) -> pd.Series:
    pieces: list[pd.Series] = []
    for path in paths:
        ds = xr.open_dataset(path, engine="h5netcdf")
        ts = pd.to_datetime(ds["time"].values)
        y = np.asarray(ds["xrsb_flux"].values, dtype=float)
        s = pd.Series(y, index=ts)
        pieces.append(s)
        ds.close()
    out = pd.concat(pieces).sort_index()
    return out[~out.index.duplicated(keep="first")]


def standardize_curve(segment: np.ndarray, m_grid: int = M_GRID) -> np.ndarray | None:
    if segment.size < MIN_DURATION_MIN:
        return None
    y = segment.astype(float)
    if APPLY_INTERPOLATION:
        t = np.linspace(0, 1, len(y))
        t_ref = np.linspace(0, 1, m_grid)
        y_ref = np.interp(t_ref, t, y)
    else:
        if len(y) != m_grid:
            return None
        t_ref = np.linspace(0, 1, m_grid)
        y_ref = y

    y_ref = y_ref - np.trapezoid(y_ref, t_ref)
    if APPLY_L2_NORMALIZATION:
        l2 = float(np.sqrt(np.trapezoid(y_ref**2, t_ref)))
        if l2 < 1e-12:
            return None
        return y_ref / l2
    return y_ref

In [ ]:
flare_url = (
    "https://data.ngdc.noaa.gov/platforms/solar-space-observing-satellites/goes/"
    "multi/l2/data/xrsf-l2-flrpt_science/csv/"
    "sci_xrsf-l2-flrpt_geo_s19950103_e20260412_v1-0-0.csv"
)

flare_path = download(flare_url, DATA_DIR / "flare_report.csv")
flares = pd.read_csv(flare_path, parse_dates=["time", "start_time", "end_time"])
flares = flares.dropna(subset=["start_time", "end_time", "xrsb_irrad_source"])
flares = flares[flares["xrsb_irrad_source"].isin(TARGET_SATELLITES)]

flares["duration_min"] = (
    (flares["end_time"] - flares["start_time"]).dt.total_seconds() / 60.0
)
flares = flares[
    (flares["duration_min"] >= MIN_DURATION_MIN)
    & (flares["duration_min"] <= MAX_DURATION_MIN)
]

urls18 = available_yearly_urls("18")
urls19 = available_yearly_urls("19")
available = {
    "GOES-18": sorted(urls18.keys()),
    "GOES-19": sorted(urls19.keys()),
}

common_start = max(min(available["GOES-18"]), min(available["GOES-19"]))
common_end = min(max(available["GOES-18"]), max(available["GOES-19"]))
target_start = max(common_start, common_end - (N_YEARS_TARGET - 1))

flares = flares[
    (flares["start_time"].dt.year >= target_start)
    & (flares["start_time"].dt.year <= common_end)
]

print(f"Common GOES-18/19 coverage: {common_start}-{common_end}")
print(f"Requested window: {N_YEARS_TARGET} years")
print(f"Using available shared window: {target_start}-{common_end}")
print(f"Events in window before sampling: {len(flares):,}")

sampled = []
for sat, group in flares.groupby("xrsb_irrad_source", sort=False):
    n_take = min(len(group), MAX_EVENTS_PER_SATELLITE)
    sampled.append(group.sample(n=n_take, random_state=SEED))
flares = pd.concat(sampled, ignore_index=True).sort_values("start_time").reset_index(drop=True)

print(f"Events after per-satellite cap: {len(flares):,}")
flares[["xrsb_irrad_source", "start_time", "end_time", "flare_id", "flare_class"]].head()

In [ ]:
required_years_18 = sorted(
    flares.loc[flares["xrsb_irrad_source"] == "GOES-18", "start_time"].dt.year.unique().tolist()
)
required_years_19 = sorted(
    flares.loc[flares["xrsb_irrad_source"] == "GOES-19", "start_time"].dt.year.unique().tolist()
)

paths18 = [download(urls18[y], DATA_DIR / "goes18" / Path(urls18[y]).name) for y in required_years_18]
paths19 = [download(urls19[y], DATA_DIR / "goes19" / Path(urls19[y]).name) for y in required_years_19]

flux = {
    "GOES-18": load_xrsb_flux(paths18),
    "GOES-19": load_xrsb_flux(paths19),
}

curves: list[np.ndarray] = []
rows: list[dict[str, object]] = []

for _, row in flares.iterrows():
    sat = str(row["xrsb_irrad_source"])
    seg = flux[sat].loc[row["start_time"] : row["end_time"]].dropna().to_numpy()
    curve = standardize_curve(seg, m_grid=M_GRID)
    if curve is None:
        continue
    curves.append(curve)
    rows.append(
        {
            "flare_id": row["flare_id"],
            "satellite": sat,
            "flare_class": row["flare_class"],
            "start_time": row["start_time"],
            "end_time": row["end_time"],
            "duration_min": float(row["duration_min"]),
        }
    )

X = np.vstack(curves)
meta = pd.DataFrame(rows)

print(f"Final curve matrix shape: {X.shape}")
meta.head()

In [ ]:
selected_r, labels, centers, _ = tune_r(
    X,
    n_clusters=K,
    n_restarts=N_RESTARTS,
    seed=SEED,
)

meta = meta.copy()
meta["cluster"] = labels
meta.to_csv(RESULTS_DIR / "goes_xray_cluster_assignments.csv", index=False)
np.save(RESULTS_DIR / "goes_xray_curves.npy", X)
np.save(RESULTS_DIR / "goes_xray_cluster_centers.npy", centers)

print("Clustering setup: GOES-18 and GOES-19 curves combined into one dataset")
print(f"Number of combined curves: {len(X)}")
print(f"Selected adaptive r = {selected_r:.4f}")
display(meta.groupby(["satellite", "cluster"]).size().rename("n_curves").reset_index())

In [ ]:
style.fontsize = 10
viz = apply_plotstyle()
SPAGHETTI_N_SAMPLES = 50
from matplotlib.ticker import MaxNLocator
fig, ax = plt.subplots(figsize=(6.5, 3.5))
n_plot = min(SPAGHETTI_N_SAMPLES, X.shape[0])
plot_idx = RNG.choice(X.shape[0], size=n_plot, replace=False)
X_plot = X[plot_idx]
rainbow = mpl.colormaps["rainbow"](np.linspace(0, 1, X_plot.shape[0]))

for i, y in enumerate(X_plot):
    ax.plot(GRID, y, color=rainbow[i], linewidth=0.4, alpha=1)

ax.axhline(0.0, color="black", linewidth=0.75, linestyle="--")
ax.set_xlabel("Standardized event time")
ax.set_ylabel("Standardized X-ray flux")
ax.set_title(f"GOES X-ray flare curves (n = {X.shape[0]})")
#ax.set_ylim(-2.5, 2)
ax.set_xlim(0, 1)
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
fig.tight_layout()
fig.savefig(PLOT_DIR / "goes_spaghetti_rainbow_subset.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Cluster-specific curve plots after GEC fitting
style.fontsize = 16
viz = apply_plotstyle()
CLUSTER_PLOT_N_PER_CLUSTER = 200
fig, axes = plt.subplots(1, K, figsize=(11, 4.5), sharey=True)
from matplotlib.ticker import FormatStrFormatter, MaxNLocator
if K == 1:
    axes = [axes]

for k, ax in enumerate(axes):
    ax.set_box_aspect(1)
    mask = labels == k
    Xk = X[mask]
    color = "#2b36b2c2"
    n_plot_k = min(CLUSTER_PLOT_N_PER_CLUSTER, Xk.shape[0])
    idx_k = RNG.choice(Xk.shape[0], size=n_plot_k, replace=False)
    Xk_plot = Xk[idx_k]
    for y in Xk_plot:
        ax.plot(GRID, y, color=color, alpha=0.1, linewidth=0.4)
    ax.plot(GRID, Xk.mean(axis=0), color=color, linewidth=2, label="Cluster mean")
    ax.plot(GRID, centers[k], color="black", linewidth=1.2, linestyle="--", label="GEC center")
    ax.axhline(0.0, color="black", linewidth=0.75, linestyle="--", alpha=0.8)
    ax.set_title(f"Cluster {k + 1} (n = {Xk.shape[0]})")
    ax.set_ylim(-2.5, 2.5)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    if k > 0:
        ax.tick_params(axis="y", which="both", left=False, labelleft=False)
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))



axes[0].set_ylabel("Standardized X-ray flux")
fig.supxlabel("Standardized event time", fontsize=plt.rcParams["axes.labelsize"], y=0.06)
handles, labels_legend = axes[0].get_legend_handles_labels()
#fig.legend(handles, labels_legend, loc="upper center", ncol=2, frameon=True)
fig.tight_layout()
fig.savefig(PLOT_DIR / "goes_cluster_specific_curves.pdf", bbox_inches="tight")
plt.show()